In [1]:
# imports and config
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import searoute as sr
import requests
import country_converter as coco
import folium
from IPython.display import display

pd.set_option("display.max_columns", 120)

In [2]:
# paths and base dimensions
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

FACT_ROOT = PROJECT_ROOT / "data" / "silver" / "comtrade" / "comtrade_fact"
FACT_FILES = sorted(FACT_ROOT.rglob("ref_year=*/reporter_iso3=*/*.parquet"))

DIMENSIONS_ROOT = PROJECT_ROOT / "data" / "silver" / "comtrade" / "dimensions"
DIMENSIONS_ROOT.mkdir(parents=True, exist_ok=True)

DIM_COUNTRY_PATH = DIMENSIONS_ROOT / "dim_country.parquet"
DIM_OUTPUT_PATH = PROJECT_ROOT / "data" / "silver" / "comtrade" / "dim_trade_routes.parquet"
COUNTRY_PORT_DIM_PATH = DIMENSIONS_ROOT / "dim_country_ports.parquet"
PORT_BASIN_DIM_PATH = DIMENSIONS_ROOT / "dim_port_basin.parquet"
ROUTE_APPLICABILITY_PATH = DIMENSIONS_ROOT / "bridge_country_route_applicability.parquet"
CHOKEPOINT_DIM_PATH = DIMENSIONS_ROOT / "dim_chokepoint.parquet"
CHOKEPOINT_RULES_PATH = DIMENSIONS_ROOT / "bridge_port_basin_chokepoints.parquet"

PORT_INDEX_PATH = PROJECT_ROOT / "ingest" / "Kaggle" / "world_port_index" / "UpdatedPub150.csv"

if not DIM_COUNTRY_PATH.exists():
    raise FileNotFoundError(f"dim_country not found: {DIM_COUNTRY_PATH}")
if not PORT_INDEX_PATH.exists():
    raise FileNotFoundError(f"World Port Index file not found: {PORT_INDEX_PATH}")
if not FACT_FILES:
    raise FileNotFoundError(f"No parquet fact files found under: {FACT_ROOT}")

dim_country = pd.read_parquet(DIM_COUNTRY_PATH)

if DIM_OUTPUT_PATH.exists():
    dim_trade_routes_existing = pd.read_parquet(DIM_OUTPUT_PATH)
else:
    dim_trade_routes_existing = pd.DataFrame(
        columns=[
            "reporter_iso3",
            "partner_iso3",
            "partner2_iso3",
            "reporter_port",
            "partner2_port",
            "partner_port",
            "reporter_basin",
            "partner_basin",
            "distance_km",
            "sea_distance_km",
            "sea_distance_direct_km",
            "sea_distance_forced_km",
            "main_chokepoint",
            "route_group",
            "route_mode",
            "route_basis",
            "route_confidence",
            "route_applicability_status",
            "mot_codes_seen",
            "route_path_coords",
        ]
    )

display(dim_country.head(3))
display(dim_trade_routes_existing.head(3))

,country_code,iso3,country_name,region,subregion,continent,is_eu,is_oecd
0,473,A79,"LAIA, nes",Americas,Latin America and the Caribbean,Americas,False,False
1,533,ABW,Aruba,Americas,Caribbean,North America,False,False
2,4,AFG,Afghanistan,Asia,Southern Asia,Asia,False,False


,reporter_iso3,partner_iso3,partner2_iso3,reporter_port,partner2_port,partner_port,distance_km,sea_distance_km,sea_distance_direct_km,sea_distance_forced_km,main_chokepoint,route_group,route_mode
0,BGR,AGO,None,Burgas,None,Cabinda,5558.37,10422.16,10422.16,10422.16,Gibraltar Strait,GIBRALTAR_EXPOSED,forced_chokepoint
1,BGR,ALB,None,Burgas,None,Shengjin,653.87,1572.86,1572.86,NaN,Other / Unclassified,OTHER_ROUTE,direct
2,BGR,ARE,None,Burgas,None,Abu Zaby,3169.30,7275.16,7275.16,7275.16,Suez Canal,SUEZ_EXPOSED,forced_chokepoint


In [3]:
# build a route applicability bridge from fact motCode values
silver_df = pd.concat([pd.read_parquet(path) for path in FACT_FILES], ignore_index=True)

silver_grain = [
    "period",
    "reporter_iso3",
    "partner_iso3",
    "flowCode",
    "cmdCode",
    "classification_version",
    "customsCode",
    "motCode",
    "partner2Code",
]

silver_df.drop_duplicates(subset=silver_grain, inplace=True)

code_to_iso3 = {}
if "country_code" in dim_country.columns and "iso3" in dim_country.columns:
    code_to_iso3 = dict(
        zip(
            pd.to_numeric(dim_country["country_code"], errors="coerce").astype("Int64"),
            dim_country["iso3"].astype("string").str.upper().str.strip(),
        )
    )

def _normalize_iso3(series: pd.Series) -> pd.Series:
    clean = series.astype("string").str.upper().str.strip()
    clean = clean.replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "W00": pd.NA})
    valid = clean.str.fullmatch(r"[A-Z]{3}", na=False)
    return clean.where(valid, pd.NA)

if "partner2_iso3" in silver_df.columns:
    silver_df["partner2_iso3"] = silver_df["partner2_iso3"]
elif "partner2ISO" in silver_df.columns:
    silver_df["partner2_iso3"] = silver_df["partner2ISO"]
elif "partner2Code" in silver_df.columns and code_to_iso3:
    p2_num = pd.to_numeric(silver_df["partner2Code"], errors="coerce").astype("Int64")
    silver_df["partner2_iso3"] = p2_num.map(code_to_iso3)
else:
    silver_df["partner2_iso3"] = pd.NA

for col in ["reporter_iso3", "partner_iso3", "partner2_iso3"]:
    silver_df[col] = _normalize_iso3(silver_df[col])

silver_df["motCode"] = pd.to_numeric(silver_df["motCode"], errors="coerce").astype("Int64")

SEA_CODES = {2100}
INLAND_WATER_CODES = {2200}
UNKNOWN_CODES = {0, 9000, 9900}
NON_MARINE_CODES = {1000, 3100, 3200, 9100, 9110, 9200, 9300}

route_candidates = (
    silver_df
    .dropna(subset=["reporter_iso3", "partner_iso3"])
    .groupby(["reporter_iso3", "partner_iso3", "partner2_iso3"], dropna=False)
    .agg(
        row_count=("analysis_grain", "nunique"),
        trade_value_usd=("trade_value_usd", "sum"),
        has_sea=("motCode", lambda s: s.dropna().isin(SEA_CODES).any()),
        has_inland_water=("motCode", lambda s: s.dropna().isin(INLAND_WATER_CODES).any()),
        has_unknown=("motCode", lambda s: s.dropna().isin(UNKNOWN_CODES).any()),
        has_non_marine=("motCode", lambda s: s.dropna().isin(NON_MARINE_CODES).any()),
        mot_codes_seen=("motCode", lambda s: "|".join(sorted({str(int(x)) for x in s.dropna().unique()}))),
    )
    .reset_index()
)

route_candidates["route_applicability_status"] = np.select(
    [
        route_candidates["has_sea"],
        (~route_candidates["has_sea"]) & route_candidates["has_inland_water"],
        (~route_candidates["has_sea"]) & (~route_candidates["has_inland_water"]) & route_candidates["has_unknown"],
        (~route_candidates["has_sea"]) & (~route_candidates["has_inland_water"]) & route_candidates["has_non_marine"],
    ],
    [
        "MARITIME_ELIGIBLE",
        "INLAND_WATER_ONLY",
        "UNKNOWN_ONLY",
        "NON_MARITIME_ONLY",
    ],
    default="NO_MODE_EVIDENCE",
)

route_candidates.to_parquet(ROUTE_APPLICABILITY_PATH, index=False)

display(route_candidates["route_applicability_status"].value_counts(dropna=False))
display(route_candidates.head(10))

route_applicability_status
UNKNOWN_ONLY         3274
MARITIME_ELIGIBLE    1688
INLAND_WATER_ONLY      25
Name: count, dtype: int64

,reporter_iso3,partner_iso3,partner2_iso3,row_count,trade_value_usd,has_sea,has_inland_water,has_unknown,has_non_marine,mot_codes_seen,route_applicability_status
0,BGR,AGO,<NA>,16,1.117763e+06,True,False,True,True,0|1000|2100,MARITIME_ELIGIBLE
1,BGR,ALB,<NA>,528,6.187100e+08,True,False,True,True,0|1000|2100|3200,MARITIME_ELIGIBLE
2,BGR,ARE,<NA>,250,1.508292e+08,True,True,True,True,0|1000|2100|2200|3200,MARITIME_ELIGIBLE
3,BGR,ARG,<NA>,341,1.413164e+07,True,False,True,True,0|1000|2100|3100|3200,MARITIME_ELIGIBLE
4,BGR,ARM,<NA>,238,1.966293e+07,True,False,True,True,0|2100|3200,MARITIME_ELIGIBLE
5,BGR,ATG,<NA>,24,5.069904e+05,True,False,True,True,0|2100|3200,MARITIME_ELIGIBLE
6,BGR,AUS,<NA>,106,3.912566e+06,True,False,True,True,0|1000|2100|3200,MARITIME_ELIGIBLE
7,BGR,AUT,<NA>,1890,3.797258e+08,False,True,True,True,0|1000|2200|3100|3200|9200,INLAND_WATER_ONLY
8,BGR,AZE,<NA>,250,6.370388e+08,True,False,True,True,0|1000|2100|3100|3200,MARITIME_ELIGIBLE
9,BGR,BEL,<NA>,1394,6.942962e+08,True,False,True,True,0|1000|2100|3100|3200|9200,MARITIME_ELIGIBLE


In [4]:
# build dim_country_ports and dim_port_basin from World Port Index using inferred basin logic
port_raw = pd.read_csv(PORT_INDEX_PATH, low_memory=False)

port_candidates = port_raw.rename(
    columns={
        "Main Port Name": "main_port_name",
        "Alternate Port Name": "alternate_port_name",
        "Country Code": "country_label",
        "World Water Body": "world_water_body",
        "Harbor Size": "harbor_size",
        "Harbor Type": "harbor_type",
        "Harbor Use": "harbor_use",
        "Facilities - Container": "fac_container",
        "Facilities - Solid Bulk": "fac_solid_bulk",
        "Facilities - Liquid Bulk": "fac_liquid_bulk",
        "Facilities - Oil Terminal": "fac_oil_terminal",
        "Facilities - LNG Terminal": "fac_lng_terminal",
        "Latitude": "latitude",
        "Longitude": "longitude",
    }
).copy()

for col in [
    "main_port_name",
    "alternate_port_name",
    "country_label",
    "world_water_body",
    "harbor_size",
    "harbor_type",
    "harbor_use",
]:
    port_candidates[col] = port_candidates[col].astype("string").str.strip()

port_candidates["port_name"] = port_candidates["main_port_name"].fillna(port_candidates["alternate_port_name"])
port_candidates["latitude"] = pd.to_numeric(port_candidates["latitude"], errors="coerce")
port_candidates["longitude"] = pd.to_numeric(port_candidates["longitude"], errors="coerce")

port_candidates["iso3"] = coco.convert(
    names=port_candidates["country_label"].fillna(""),
    to="ISO3",
    not_found="not found",
)
port_candidates["iso3"] = pd.Series(port_candidates["iso3"], index=port_candidates.index).replace("not found", pd.NA)

def _as_txt(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()

def infer_port_basin(world_water_body: str) -> str:
    txt = _as_txt(world_water_body)

    # Black Sea and direct exits
    if "black sea" in txt or "sea of azov" in txt:
        return "BLACK_SEA"

    # Mediterranean family
    if (
        "mediterranean" in txt
        or "gulf of lion" in txt
        or "adriatic" in txt
        or "aegean" in txt
        or "ionian" in txt
        or "tyrrhenian" in txt
        or "sea of marmara" in txt
    ):
        return "MEDITERRANEAN"

    # Gulf family
    if "persian gulf" in txt or "arabian gulf" in txt:
        return "GULF"

    # Red Sea and adjacent
    if "red sea" in txt or "gulf of aqaba" in txt or "gulf of suez" in txt:
        return "RED_SEA"

    # Baltic
    if "baltic" in txt or "gulf of bothnia" in txt or "gulf of finland" in txt:
        return "BALTIC"

    # Northern Europe Atlantic-facing seas
    if (
        "north sea" in txt
        or "english channel" in txt
        or "bay of biscay" in txt
        or "irish sea" in txt
        or "celtic sea" in txt
        or "norwegian sea" in txt
    ):
        return "NORTH_ATLANTIC_EUROPE"

    # Atlantic family
    if "north atlantic ocean" in txt or "south atlantic ocean" in txt:
        return "ATLANTIC"
    if "gulf of mexico" in txt or "caribbean" in txt:
        return "ATLANTIC"

    # Indian Ocean family
    if "indian ocean" in txt or "arabian sea" in txt or "bay of bengal" in txt:
        return "INDIAN_OCEAN"

    # Western Pacific family
    if (
        "sea of japan" in txt
        or "east china sea" in txt
        or "south china sea" in txt
        or "yellow sea" in txt
        or "philippine sea" in txt
    ):
        return "WESTERN_PACIFIC"

    # Wider Pacific and arctic families
    if "north pacific ocean" in txt or "south pacific ocean" in txt or "bering sea" in txt or "sea of okhotsk" in txt or "tatar strait" in txt:
        return "PACIFIC"

    if "arctic ocean" in txt or "barents sea" in txt or "white sea" in txt or "east siberian sea" in txt:
        return "ARCTIC"

    # Inland special cases
    if "great lakes" in txt:
        return "GREAT_LAKES"
    if "danube" in txt or "river" in txt:
        return "INLAND_WATERWAY"

    return "OTHER"

port_candidates["port_basin"] = port_candidates["world_water_body"].map(infer_port_basin)

size_rank = {"Very Small": 1, "Small": 2, "Medium": 3, "Large": 4}
yes_rank = {"Yes": 1, "No": 0, "Unknown": 0}

for col in ["fac_container", "fac_solid_bulk", "fac_liquid_bulk", "fac_oil_terminal", "fac_lng_terminal"]:
    port_candidates[col] = port_candidates[col].map(yes_rank).fillna(0).astype(int)

port_candidates["size_rank"] = port_candidates["harbor_size"].map(size_rank).fillna(0).astype(int)
port_candidates["port_score"] = (
    port_candidates["size_rank"] * 10
    + port_candidates["fac_container"] * 5
    + port_candidates["fac_solid_bulk"] * 4
    + port_candidates["fac_liquid_bulk"] * 4
    + port_candidates["fac_oil_terminal"] * 6
    + port_candidates["fac_lng_terminal"] * 6
)

usable_ports = (
    port_candidates
    .dropna(subset=["iso3", "port_name", "latitude", "longitude"])
    .drop_duplicates(subset=["iso3", "port_name", "latitude", "longitude"])
    .copy()
)

MULTI_BASIN_OVERRIDES = {"RUS", "USA", "CHN", "FRA", "ESP", "TUR", "CAN", "AUS"}

def max_ports_for_country(iso3: str) -> int:
    if iso3 in MULTI_BASIN_OVERRIDES:
        return 10
    return 4

def max_ports_per_basin(iso3: str) -> int:
    if iso3 in MULTI_BASIN_OVERRIDES:
        return 2
    return 1

usable_ports = usable_ports.sort_values(
    ["iso3", "port_basin", "port_score", "size_rank", "port_name"],
    ascending=[True, True, False, False, True],
).copy()

def _retain_port_subset(group: pd.DataFrame) -> pd.DataFrame:
    iso3 = group["iso3"].iloc[0]

    per_basin = (
        group.groupby("port_basin", group_keys=False)
        .head(max_ports_per_basin(iso3))
        .copy()
    )

    if len(per_basin) >= max_ports_for_country(iso3):
        keep = per_basin.sort_values(
            ["port_score", "size_rank", "port_name"],
            ascending=[False, False, True],
        ).head(max_ports_for_country(iso3)).copy()
    else:
        already = set(per_basin.index)
        remainder = group.loc[~group.index.isin(already)].sort_values(
            ["port_score", "size_rank", "port_name"],
            ascending=[False, False, True],
        )
        n_extra = max_ports_for_country(iso3) - len(per_basin)
        keep = pd.concat([per_basin, remainder.head(n_extra)], ignore_index=False)

    keep = keep.sort_values(
        ["port_score", "size_rank", "port_basin", "port_name"],
        ascending=[False, False, True, True],
    ).copy()
    return keep

usable_ports = (
    usable_ports.groupby("iso3", group_keys=False)
    .apply(_retain_port_subset)
    .copy()
)

usable_ports["country_port_rank"] = (
    usable_ports.sort_values(["iso3", "port_score", "size_rank", "port_basin", "port_name"], ascending=[True, False, False, True, True])
    .groupby("iso3")
    .cumcount() + 1
)

dim_country_base = dim_country[["iso3", "country_name"]].drop_duplicates().copy()
dim_country_ports = dim_country_base.merge(
    usable_ports[
        [
            "iso3",
            "country_port_rank",
            "port_name",
            "latitude",
            "longitude",
            "harbor_size",
            "harbor_type",
            "harbor_use",
            "world_water_body",
            "port_basin",
            "port_score",
            "fac_container",
            "fac_solid_bulk",
            "fac_liquid_bulk",
            "fac_oil_terminal",
            "fac_lng_terminal",
        ]
    ].rename(columns={"country_port_rank": "port_rank"}),
    on="iso3",
    how="left",
)

dim_country_ports.to_parquet(COUNTRY_PORT_DIM_PATH, index=False)

dim_port_basin = (
    usable_ports[
        [
            "iso3",
            "port_name",
            "world_water_body",
            "port_basin",
            "latitude",
            "longitude",
            "port_score",
        ]
    ]
    .drop_duplicates()
    .sort_values(["iso3", "port_basin", "port_score", "port_name"], ascending=[True, True, False, True])
    .reset_index(drop=True)
)

dim_port_basin.to_parquet(PORT_BASIN_DIM_PATH, index=False)

display(dim_country_ports.head(20))
display(dim_port_basin.loc[dim_port_basin["iso3"].eq("ROU")].head(20))
display(dim_port_basin.loc[dim_port_basin["iso3"].eq("RUS")].head(20))
display(dim_port_basin.loc[dim_port_basin["iso3"].eq("TUR")].head(20))


Wake Island not found in regex
Johnson Atoll not found in regex
Midway Islands not found in regex
 not found in regex
 not found in regex
 not found in regex
 not found in regex


,iso3,country_name,port_rank,port_name,latitude,longitude,harbor_size,harbor_type,harbor_use,world_water_body,port_basin,port_score,fac_container,fac_solid_bulk,fac_liquid_bulk,fac_oil_terminal,fac_lng_terminal
0,A79,"LAIA, nes",NaN,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,Aruba,1.0,Sint Nicolaas Baai,12.433333,-69.916667,Small,Coastal (Breakwater),Unknown,Caribbean Sea; North Atlantic Ocean,ATLANTIC,20.0,0.0,0.0,0.0,0.0,0.0
2,ABW,Aruba,2.0,Paarden Baai - (Oranjestad),12.516667,-70.033333,Very Small,Coastal (Breakwater),Unknown,Caribbean Sea; North Atlantic Ocean,ATLANTIC,10.0,0.0,0.0,0.0,0.0,0.0
3,AFG,Afghanistan,NaN,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AGO,Angola,1.0,Cabinda,-5.533333,12.200000,Small,Open Roadstead,Unknown,South Atlantic Ocean,OTHER,20.0,0.0,0.0,0.0,0.0,0.0
5,AGO,Angola,2.0,Estrela Oil Field,-6.450000,12.400000,Small,Open Roadstead,Unknown,South Atlantic Ocean,OTHER,20.0,0.0,0.0,0.0,0.0,0.0
6,AGO,Angola,3.0,Lobito,-12.333333,13.566667,Small,Coastal (Natural),Unknown,South Atlantic Ocean,OTHER,20.0,0.0,0.0,0.0,0.0,0.0
7,AIA,Anguilla,NaN,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ALB,Albania,1.0,Durres,41.316667,19.450000,Small,Coastal (Breakwater),Unknown,Adriatic Sea; Mediterranean Sea; North Atlanti...,MEDITERRANEAN,20.0,0.0,0.0,0.0,0.0,0.0
9,ALB,Albania,2.0,Shengjin,41.816667,19.600000,Very Small,Coastal (Natural),Unknown,Adriatic Sea; Mediterranean Sea; North Atlanti...,MEDITERRANEAN,10.0,0.0,0.0,0.0,0.0,0.0


,iso3,port_name,world_water_body,port_basin,latitude,longitude,port_score
395,ROU,Galati,Black Sea; North Atlantic Ocean,BLACK_SEA,45.416667,28.083333,33
396,ROU,Constanta,Black Sea; North Atlantic Ocean,BLACK_SEA,44.166667,28.650000,30
397,ROU,Danube-Black Sea Canal,Black Sea; North Atlantic Ocean,BLACK_SEA,44.266667,28.183333,30


,iso3,port_name,world_water_body,port_basin,latitude,longitude,port_score
398,RUS,Vladivostok,Sea of Japan; North Pacific Ocean,WESTERN_PACIFIC,43.108889,131.888333,59
399,RUS,Kaliningrad,Baltic Sea; North Atlantic Ocean,BALTIC,54.700000,20.483333,43
400,RUS,Vyborg,Gulf of Finland; Baltic Sea; North Atlantic Ocean,BALTIC,60.716667,28.750000,43
401,RUS,Murmansk,Barents Sea; Arctic Ocean,ARCTIC,68.983333,33.050000,40
402,RUS,Novorossiysk,Black Sea; North Atlantic Ocean,BLACK_SEA,44.716667,37.783333,40
403,RUS,Sankt-Peterburg,Gulf of Finland; Baltic Sea; North Atlantic Ocean,BALTIC,59.933333,30.300000,40
404,RUS,Arkhangels'k,White Sea; Barents Sea; Arctic Ocean,ARCTIC,64.529722,40.521667,30
405,RUS,De Kastri,Tatar Strait; North Pacific Ocean,PACIFIC,51.466667,140.783333,30


In [6]:
# coverage check for ports and route applicability
country_port_coverage = (
    dim_country_ports.groupby(["iso3", "country_name"], as_index=False)["port_rank"]
    .count()
    .rename(columns={"port_rank": "ports_found"})
)

country_geo = dim_country[["iso3", "country_name", "continent", "region", "subregion"]].drop_duplicates().copy()
country_geo = country_geo.merge(country_port_coverage[["iso3", "ports_found"]], on="iso3", how="left")
country_geo["ports_found"] = country_geo["ports_found"].fillna(0).astype(int)

KNOWN_LANDLOCKED = {
    "AUT", "CHE", "CZE", "HUN", "SVK", "LUX", "MDA", "SRB", "MKD",
    "BLR", "ARM", "KAZ", "KGZ", "MNG", "NPL", "AFG", "BOL", "PRY",
    "ETH", "UGA", "RWA", "BDI", "MLI", "NER", "TCD", "ZMB", "ZWE","LSO"
}

country_geo["is_landlocked_assumed"] = country_geo["iso3"].isin(KNOWN_LANDLOCKED)
country_geo["has_wpi_ports"] = country_geo["ports_found"] > 0

COUNTRY_FLAGS = country_geo.set_index("iso3")[["has_wpi_ports", "is_landlocked_assumed"]].to_dict("index")

display(country_geo.sort_values(["has_wpi_ports", "ports_found", "country_name"]).head(30))

,iso3,country_name,continent,region,subregion,ports_found,is_landlocked_assumed,has_wpi_ports
2,AFG,Afghanistan,Asia,Asia,Southern Asia,0,True,False
6,AND,Andorra,Europe,Europe,Southern Europe,0,False,False
4,AIA,Anguilla,North America,Americas,Caribbean,0,False,False
238,_X,"Areas, nes",Special,Special,"Areas, nes",0,False,False
9,ARM,Armenia,Asia,Asia,Western Asia,0,True,False
15,AUT,Austria,Europe,Europe,Western Europe,0,True,False
16,AZE,Azerbaijan,Asia,Asia,Western Asia,0,False,False
28,BLR,Belarus,Europe,Europe,Eastern Europe,0,True,False
35,BTN,Bhutan,Asia,Asia,Southern Asia,0,False,False
31,BOL,Bolivia (Plurinational State of),South America,Americas,South America,0,True,False


In [7]:
# persist chokepoint master data and derive basin-to-basin forcing rules from a basin graph
dim_chokepoint = pd.DataFrame(
    [
        {"chokepoint_name": "Turkish Straits", "longitude": 29.0, "latitude": 41.0, "kind": "strait"},
        {"chokepoint_name": "Suez Canal", "longitude": 32.5498, "latitude": 30.8167, "kind": "canal"},
        {"chokepoint_name": "Hormuz Strait", "longitude": 56.25, "latitude": 26.57, "kind": "strait"},
        {"chokepoint_name": "Bab el-Mandeb", "longitude": 43.33, "latitude": 12.58, "kind": "strait"},
        {"chokepoint_name": "Panama Canal", "longitude": -79.58, "latitude": 9.08, "kind": "canal"},
        {"chokepoint_name": "Malacca Strait", "longitude": 99.8, "latitude": 2.5, "kind": "strait"},
        {"chokepoint_name": "Gibraltar Strait", "longitude": -5.6, "latitude": 35.95, "kind": "strait"},
        {"chokepoint_name": "Cape of Good Hope", "longitude": 18.47, "latitude": -34.36, "kind": "cape"},
    ]
)
dim_chokepoint.to_parquet(CHOKEPOINT_DIM_PATH, index=False)

CHOKEPOINT_COORDS = {
    row.chokepoint_name: (float(row.longitude), float(row.latitude))
    for row in dim_chokepoint.itertuples(index=False)
}

# basin graph:
# each edge represents two basins that are adjacent in shipping terms.
# chokepoint_name is only populated when that adjacency requires a major chokepoint.
BASIN_GRAPH_EDGES = [
    ("BLACK_SEA", "MEDITERRANEAN", "Turkish Straits"),
    ("MEDITERRANEAN", "RED_SEA", "Suez Canal"),
    ("RED_SEA", "INDIAN_OCEAN", "Bab el-Mandeb"),
    ("GULF", "INDIAN_OCEAN", "Hormuz Strait"),
    ("INDIAN_OCEAN", "WESTERN_PACIFIC", "Malacca Strait"),
    ("MEDITERRANEAN", "ATLANTIC", "Gibraltar Strait"),
    ("MEDITERRANEAN", "NORTH_ATLANTIC_EUROPE", "Gibraltar Strait"),
    ("ATLANTIC", "WESTERN_PACIFIC", "Panama Canal"),
    ("ATLANTIC", "INDIAN_OCEAN", "Cape of Good Hope"),
    ("NORTH_ATLANTIC_EUROPE", "ATLANTIC", None),
    ("BALTIC", "NORTH_ATLANTIC_EUROPE", None),
    ("ARCTIC", "NORTH_ATLANTIC_EUROPE", None),
    ("PACIFIC", "WESTERN_PACIFIC", None),
    ("WESTERN_PACIFIC", "INDIAN_OCEAN", "Malacca Strait"),
]

BASIN_ADJ = {}
for origin_basin, destination_basin, chokepoint_name in BASIN_GRAPH_EDGES:
    BASIN_ADJ.setdefault(origin_basin, []).append((destination_basin, chokepoint_name))
    BASIN_ADJ.setdefault(destination_basin, []).append((origin_basin, chokepoint_name))

def _shortest_basin_path(origin_basin: str, destination_basin: str) -> list[str]:
    if pd.isna(origin_basin) or pd.isna(destination_basin):
        return []

    if origin_basin == destination_basin:
        return []

    from collections import deque

    queue = deque([(origin_basin, [])])
    visited = {origin_basin}

    while queue:
        basin, cp_path = queue.popleft()

        for next_basin, chokepoint_name in BASIN_ADJ.get(basin, []):
            if next_basin in visited:
                continue

            next_path = cp_path.copy()
            if chokepoint_name is not None:
                next_path.append(chokepoint_name)

            if next_basin == destination_basin:
                return next_path

            visited.add(next_basin)
            queue.append((next_basin, next_path))

    return []

known_basins = sorted(set(dim_port_basin["port_basin"].dropna().unique()) | set(BASIN_ADJ.keys()))

bridge_records = []
for origin_basin in known_basins:
    for destination_basin in known_basins:
        if origin_basin == destination_basin:
            continue
        cp_path = _shortest_basin_path(origin_basin, destination_basin)
        for leg_order, chokepoint_name in enumerate(cp_path, start=1):
            bridge_records.append(
                {
                    "origin_basin": origin_basin,
                    "destination_basin": destination_basin,
                    "leg_order": leg_order,
                    "chokepoint_name": chokepoint_name,
                }
            )

bridge_port_basin_chokepoints = pd.DataFrame(bridge_records).sort_values(
    ["origin_basin", "destination_basin", "leg_order"]
).reset_index(drop=True)

bridge_port_basin_chokepoints.to_parquet(CHOKEPOINT_RULES_PATH, index=False)

BASIN_RULES = (
    bridge_port_basin_chokepoints
    .sort_values(["origin_basin", "destination_basin", "leg_order"])
    .groupby(["origin_basin", "destination_basin"])["chokepoint_name"]
    .apply(list)
    .to_dict()
)

display(dim_chokepoint)
display(bridge_port_basin_chokepoints.head(30))


,chokepoint_name,longitude,latitude,kind
0,Turkish Straits,29.0000,41.0000,strait
1,Suez Canal,32.5498,30.8167,canal
2,Hormuz Strait,56.2500,26.5700,strait
3,Bab el-Mandeb,43.3300,12.5800,strait
4,Panama Canal,-79.5800,9.0800,canal
5,Malacca Strait,99.8000,2.5000,strait
6,Gibraltar Strait,-5.6000,35.9500,strait
7,Cape of Good Hope,18.4700,-34.3600,cape


,origin_basin,destination_basin,leg_order,chokepoint_name
0,BLACK_SEA,MEDITERRANEAN,1,Turkish Straits
1,BLACK_SEA,GULF,1,Turkish Straits
2,BLACK_SEA,GULF,2,Suez Canal
3,BLACK_SEA,GULF,3,Bab el-Mandeb
4,BLACK_SEA,GULF,4,Hormuz Strait
5,BLACK_SEA,INDIAN_OCEAN,1,Turkish Straits
6,BLACK_SEA,INDIAN_OCEAN,2,Suez Canal
7,BLACK_SEA,INDIAN_OCEAN,3,Bab el-Mandeb
8,BLACK_SEA,WESTERN_PACIFIC,1,Turkish Straits
9,BLACK_SEA,WESTERN_PACIFIC,2,Suez Canal


In [8]:
# extract maritime-eligible route keys for a clean rebuild of the dimension
FULL_REBUILD_DIM_TRADE_ROUTES = True

existing_keys = set()
if (not FULL_REBUILD_DIM_TRADE_ROUTES) and (not dim_trade_routes_existing.empty):
    existing_keys = set(
        dim_trade_routes_existing[["reporter_iso3", "partner_iso3", "partner2_iso3"]]
        .fillna("__NULL__")
        .itertuples(index=False, name=None)
    )

to_route = route_candidates[
    route_candidates["route_applicability_status"] == "MARITIME_ELIGIBLE"
].copy()

to_route["_key"] = list(
    to_route[["reporter_iso3", "partner_iso3", "partner2_iso3"]]
    .fillna("__NULL__")
    .itertuples(index=False, name=None)
)

if existing_keys:
    to_route = to_route[~to_route["_key"].isin(existing_keys)].copy()

display(to_route.head(10))
print("Maritime route keys to build:", len(to_route))
print("Full rebuild enabled:", FULL_REBUILD_DIM_TRADE_ROUTES)


,reporter_iso3,partner_iso3,partner2_iso3,row_count,trade_value_usd,has_sea,has_inland_water,has_unknown,has_non_marine,mot_codes_seen,route_applicability_status,_key
351,ESP,AGO,AGO,232,3.197270e+10,True,False,True,True,0|1000|2100,MARITIME_ELIGIBLE,"(ESP, AGO, AGO)"
352,ESP,AGO,MEX,4,5.026263e+07,True,False,True,False,0|2100,MARITIME_ELIGIBLE,"(ESP, AGO, MEX)"
354,ESP,ALB,ALB,310,4.245283e+09,True,False,True,True,0|1000|2100|3200,MARITIME_ELIGIBLE,"(ESP, ALB, ALB)"
355,ESP,ALB,ITA,16,7.326918e+05,True,False,True,True,0|2100|3200,MARITIME_ELIGIBLE,"(ESP, ALB, ITA)"
365,ESP,ARE,ARE,384,1.049302e+10,True,False,True,True,0|1000|2100|3100|3200,MARITIME_ELIGIBLE,"(ESP, ARE, ARE)"
366,ESP,ARE,AZE,4,1.417981e+05,True,False,True,False,0|2100,MARITIME_ELIGIBLE,"(ESP, ARE, AZE)"
368,ESP,ARE,BHR,4,1.272833e+05,True,False,True,False,0|2100,MARITIME_ELIGIBLE,"(ESP, ARE, BHR)"
369,ESP,ARE,GBR,4,9.110958e+07,True,False,True,False,0|2100,MARITIME_ELIGIBLE,"(ESP, ARE, GBR)"
370,ESP,ARE,IDN,4,2.659673e+06,True,False,True,False,0|2100,MARITIME_ELIGIBLE,"(ESP, ARE, IDN)"
371,ESP,ARE,MLT,12,3.074208e+08,True,False,True,False,0|2100,MARITIME_ELIGIBLE,"(ESP, ARE, MLT)"


New maritime route keys to build: 1093


In [9]:
# build dim_trade_routes using motCode-gated, basin-aware routing with clearer neighbour logic
country_ports_clean = (
    dim_country_ports
    .dropna(subset=["iso3", "port_name", "latitude", "longitude"])
    .copy()
)

COUNTRY_PORTS = (
    country_ports_clean.sort_values(["iso3", "port_rank", "port_name"])
    .groupby("iso3")
    .apply(
        lambda g: [
            {
                "port_name": row.port_name,
                "lonlat": (float(row.longitude), float(row.latitude)),
                "port_basin": row.port_basin,
                "port_score": float(row.port_score) if pd.notna(row.port_score) else 0.0,
                "world_water_body": row.world_water_body,
                "fac_container": int(row.fac_container) if pd.notna(row.fac_container) else 0,
                "fac_solid_bulk": int(row.fac_solid_bulk) if pd.notna(row.fac_solid_bulk) else 0,
                "fac_liquid_bulk": int(row.fac_liquid_bulk) if pd.notna(row.fac_liquid_bulk) else 0,
                "fac_oil_terminal": int(row.fac_oil_terminal) if pd.notna(row.fac_oil_terminal) else 0,
                "fac_lng_terminal": int(row.fac_lng_terminal) if pd.notna(row.fac_lng_terminal) else 0,
            }
            for row in g.itertuples(index=False)
        ]
    )
    .to_dict()
)

def infer_chokepoints(reporter_basin: str, partner_basin: str) -> list[str]:
    return BASIN_RULES.get((reporter_basin, partner_basin), [])

def _gc_distance_km(a_lonlat: tuple[float, float], b_lonlat: tuple[float, float]) -> float:
    lon1, lat1 = map(math.radians, a_lonlat)
    lon2, lat2 = map(math.radians, b_lonlat)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    h = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 6371.0088 * (2 * math.asin(math.sqrt(h)))

def _sea_distance_km(a_lonlat: tuple[float, float], b_lonlat: tuple[float, float]) -> float:
    route = sr.searoute(list(a_lonlat), list(b_lonlat), units="km")
    return float(route.properties["length"]), route

def _is_neighbour_candidate(gc_km: float) -> bool:
    return gc_km <= 2500.0

def _is_very_near_candidate(gc_km: float) -> bool:
    return gc_km <= 1200.0

def _pair_selection_score(a: dict, b: dict, gc_km: float, cp_sequence: list[str]) -> float:
    pair_score = 0.0

    # favour capable ports
    pair_score += a["port_score"] * 0.20
    pair_score += b["port_score"] * 0.20

    # distance still matters
    pair_score -= gc_km * 0.0015

    # same-basin and low-chokepoint paths are preferred, especially for nearby countries
    if a["port_basin"] == b["port_basin"]:
        pair_score += 7.0
    elif len(cp_sequence) == 1:
        pair_score += 2.5

    pair_score -= len(cp_sequence) * 1.75

    # neighbouring countries should almost never jump to multi-chokepoint routes
    if _is_neighbour_candidate(gc_km):
        if len(cp_sequence) == 0:
            pair_score += 6.0
        elif len(cp_sequence) == 1:
            pair_score += 1.5
        else:
            pair_score -= 8.0

    if _is_very_near_candidate(gc_km):
        if len(cp_sequence) == 0:
            pair_score += 5.0
        elif len(cp_sequence) > 1:
            pair_score -= 10.0

    return pair_score

def choose_best_port_pair(a_iso: str, b_iso: str):
    a_ports = COUNTRY_PORTS.get(a_iso, [])
    b_ports = COUNTRY_PORTS.get(b_iso, [])

    if not a_ports or not b_ports:
        return None

    best = None
    best_score = -np.inf

    for a in a_ports:
        for b in b_ports:
            gc_km = _gc_distance_km(a["lonlat"], b["lonlat"])
            cp_sequence = infer_chokepoints(a["port_basin"], b["port_basin"])
            pair_score = _pair_selection_score(a, b, gc_km, cp_sequence)

            candidate = {
                "reporter_port": a["port_name"],
                "partner_port": b["port_name"],
                "reporter_lonlat": a["lonlat"],
                "partner_lonlat": b["lonlat"],
                "reporter_basin": a["port_basin"],
                "partner_basin": b["port_basin"],
                "distance_km": gc_km,
                "cp_sequence": cp_sequence,
                "pair_score": pair_score,
            }

            if pair_score > best_score:
                best = candidate
                best_score = pair_score

    return best

def _forced_path_distance(start_lonlat: tuple[float, float], end_lonlat: tuple[float, float], chokepoints: list[str]):
    if not chokepoints:
        dist_km, route = _sea_distance_km(start_lonlat, end_lonlat)
        return dist_km, route["geometry"]["coordinates"]

    path_nodes = [start_lonlat] + [CHOKEPOINT_COORDS[cp] for cp in chokepoints if cp in CHOKEPOINT_COORDS] + [end_lonlat]
    if len(path_nodes) < 2:
        return np.nan, []

    total_km = 0.0
    stitched = []
    for i in range(len(path_nodes) - 1):
        leg_km, leg_route = _sea_distance_km(path_nodes[i], path_nodes[i + 1])
        total_km += leg_km
        leg_coords = leg_route["geometry"]["coordinates"]
        if i == 0:
            stitched.extend(leg_coords)
        else:
            stitched.extend(leg_coords[1:])
    return total_km, stitched

def _route_group(main_chokepoint: str) -> str:
    if not main_chokepoint:
        return "DIRECT_OR_UNCLASSIFIED"

    label = main_chokepoint.lower()
    if "suez" in label:
        return "SUEZ_EXPOSED"
    if "hormuz" in label:
        return "HORMUZ_EXPOSED"
    if "cape" in label:
        return "CAPE_DIVERSION_RISK"
    if "panama" in label:
        return "PANAMA_EXPOSED"
    if "malacca" in label:
        return "MALACCA_EXPOSED"
    if "gibraltar" in label:
        return "GIBRALTAR_EXPOSED"
    if "turkish" in label:
        return "BLACK_SEA_EXIT_EXPOSED"
    return "OTHER_ROUTE"

records = []

for row in to_route.itertuples(index=False):
    r_iso = row.reporter_iso3
    p_iso = row.partner_iso3
    p2_iso = row.partner2_iso3 if pd.notna(row.partner2_iso3) else None

    reporter_flags = COUNTRY_FLAGS.get(r_iso, {})
    partner_flags = COUNTRY_FLAGS.get(p_iso, {})

    if reporter_flags.get("is_landlocked_assumed", False) or partner_flags.get("is_landlocked_assumed", False):
        continue

    if r_iso not in COUNTRY_PORTS or p_iso not in COUNTRY_PORTS:
        continue

    best = choose_best_port_pair(r_iso, p_iso)
    if best is None:
        continue

    sea_distance_direct_km = np.nan
    sea_distance_forced_km = np.nan
    route_path_coords = []

    try:
        sea_distance_direct_km, sea_route = _sea_distance_km(best["reporter_lonlat"], best["partner_lonlat"])
        route_path_coords = sea_route["geometry"]["coordinates"]
    except Exception:
        pass

    cp_sequence = best["cp_sequence"]

    if cp_sequence:
        try:
            sea_distance_forced_km, forced_coords = _forced_path_distance(
                best["reporter_lonlat"],
                best["partner_lonlat"],
                cp_sequence,
            )
            if forced_coords:
                route_path_coords = forced_coords
        except Exception:
            sea_distance_forced_km = np.nan

    # choose the more plausible path:
    # nearby routes should not be forced through long multi-leg chokepoint paths
    if cp_sequence and np.isfinite(sea_distance_forced_km):
        if _is_neighbour_candidate(best["distance_km"]) and len(cp_sequence) > 1:
            sea_distance_km = sea_distance_direct_km
            route_mode = "direct"
            main_chokepoint = None
            route_confidence = "low"
        elif np.isfinite(sea_distance_direct_km) and sea_distance_forced_km > sea_distance_direct_km * 1.6 and _is_neighbour_candidate(best["distance_km"]):
            sea_distance_km = sea_distance_direct_km
            route_mode = "direct"
            main_chokepoint = None
            route_confidence = "low"
        else:
            sea_distance_km = sea_distance_forced_km
            route_mode = "forced_chokepoint"
            main_chokepoint = cp_sequence[0]
            route_confidence = "medium" if len(cp_sequence) <= 2 else "medium_low"
    else:
        sea_distance_km = sea_distance_direct_km
        route_mode = "direct"
        main_chokepoint = None
        route_confidence = "low"

    records.append(
        {
            "reporter_iso3": r_iso,
            "partner_iso3": p_iso,
            "partner2_iso3": p2_iso,
            "reporter_port": best["reporter_port"],
            "partner2_port": None,
            "partner_port": best["partner_port"],
            "reporter_basin": best["reporter_basin"],
            "partner_basin": best["partner_basin"],
            "distance_km": round(best["distance_km"], 2),
            "sea_distance_km": round(sea_distance_km, 2) if np.isfinite(sea_distance_km) else np.nan,
            "sea_distance_direct_km": round(sea_distance_direct_km, 2) if np.isfinite(sea_distance_direct_km) else np.nan,
            "sea_distance_forced_km": round(sea_distance_forced_km, 2) if np.isfinite(sea_distance_forced_km) else np.nan,
            "main_chokepoint": main_chokepoint,
            "route_group": _route_group(main_chokepoint),
            "route_mode": route_mode,
            "route_basis": "SEA_OBSERVED_MOTCODE",
            "route_confidence": route_confidence,
            "route_applicability_status": row.route_applicability_status,
            "mot_codes_seen": row.mot_codes_seen,
            "route_path_coords": route_path_coords,
        }
    )

dim_trade_routes_new = pd.DataFrame.from_records(records)

display(dim_trade_routes_new.head(20))
print("New route rows built:", len(dim_trade_routes_new))


,reporter_iso3,partner_iso3,partner2_iso3,reporter_port,partner2_port,partner_port,reporter_basin,partner_basin,distance_km,sea_distance_km,sea_distance_direct_km,sea_distance_forced_km,main_chokepoint,route_group,route_mode,route_basis,route_confidence,route_applicability_status,mot_codes_seen,route_path_coords
0,ESP,AGO,AGO,Las Palmas,None,Cabinda,ATLANTIC,OTHER,4774.14,5648.64,5648.64,NaN,NaN,DIRECT_OR_UNCLASSIFIED,direct,SEA_OBSERVED_MOTCODE,low,MARITIME_ELIGIBLE,0|1000|2100,"[[-15.3, 28.3], [-16.6896, 24.657], [-18, 21],..."
1,ESP,AGO,MEX,Las Palmas,None,Cabinda,ATLANTIC,OTHER,4774.14,5648.64,5648.64,NaN,NaN,DIRECT_OR_UNCLASSIFIED,direct,SEA_OBSERVED_MOTCODE,low,MARITIME_ELIGIBLE,0|2100,"[[-15.3, 28.3], [-16.6896, 24.657], [-18, 21],..."
2,ESP,ALB,ALB,Barcelona,None,Durres,MEDITERRANEAN,MEDITERRANEAN,1440.66,1810.40,1810.40,NaN,NaN,DIRECT_OR_UNCLASSIFIED,direct,SEA_OBSERVED_MOTCODE,low,MARITIME_ELIGIBLE,0|1000|2100|3200,"[[2.209423, 41.312166], [2.5, 41.3], [3.237316..."
3,ESP,ALB,ITA,Barcelona,None,Durres,MEDITERRANEAN,MEDITERRANEAN,1440.66,1810.40,1810.40,NaN,NaN,DIRECT_OR_UNCLASSIFIED,direct,SEA_OBSERVED_MOTCODE,low,MARITIME_ELIGIBLE,0|2100|3200,"[[2.209423, 41.312166], [2.5, 41.3], [3.237316..."
4,ESP,ARE,ARE,Barcelona,None,Mina Jabal Ali,MEDITERRANEAN,GULF,5157.19,8422.40,8422.40,8422.40,Suez Canal,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE,medium,MARITIME_ELIGIBLE,0|1000|2100|3100|3200,"[[2.209423, 41.312166], [2.5, 41.3], [2.983281..."
5,ESP,ARE,AZE,Barcelona,None,Mina Jabal Ali,MEDITERRANEAN,GULF,5157.19,8422.40,8422.40,8422.40,Suez Canal,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE,medium,MARITIME_ELIGIBLE,0|2100,"[[2.209423, 41.312166], [2.5, 41.3], [2.983281..."
6,ESP,ARE,BHR,Barcelona,None,Mina Jabal Ali,MEDITERRANEAN,GULF,5157.19,8422.40,8422.40,8422.40,Suez Canal,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE,medium,MARITIME_ELIGIBLE,0|2100,"[[2.209423, 41.312166], [2.5, 41.3], [2.983281..."
7,ESP,ARE,GBR,Barcelona,None,Mina Jabal Ali,MEDITERRANEAN,GULF,5157.19,8422.40,8422.40,8422.40,Suez Canal,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE,medium,MARITIME_ELIGIBLE,0|2100,"[[2.209423, 41.312166], [2.5, 41.3], [2.983281..."
8,ESP,ARE,IDN,Barcelona,None,Mina Jabal Ali,MEDITERRANEAN,GULF,5157.19,8422.40,8422.40,8422.40,Suez Canal,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE,medium,MARITIME_ELIGIBLE,0|2100,"[[2.209423, 41.312166], [2.5, 41.3], [2.983281..."
9,ESP,ARE,MLT,Barcelona,None,Mina Jabal Ali,MEDITERRANEAN,GULF,5157.19,8422.40,8422.40,8422.40,Suez Canal,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE,medium,MARITIME_ELIGIBLE,0|2100,"[[2.209423, 41.312166], [2.5, 41.3], [2.983281..."


New route rows built: 993


In [10]:
# merge, deduplicate, and persist the refreshed dim_trade_routes
if FULL_REBUILD_DIM_TRADE_ROUTES:
    dim_trade_routes = dim_trade_routes_new.copy()
else:
    combined = pd.concat(
        [
            dim_trade_routes_existing.reindex(columns=dim_trade_routes_new.columns),
            dim_trade_routes_new,
        ],
        ignore_index=True,
    )

    dim_trade_routes = combined.drop_duplicates(
        subset=["reporter_iso3", "partner_iso3", "partner2_iso3"],
        keep="last",
    ).sort_values(["reporter_iso3", "partner_iso3", "partner2_iso3"], na_position="last")

dim_trade_routes_export = dim_trade_routes.drop(columns=["route_path_coords"], errors="ignore").copy()
dim_trade_routes_export.to_parquet(DIM_OUTPUT_PATH, index=False)

display(dim_trade_routes["route_mode"].value_counts(dropna=False))
display(dim_trade_routes["route_group"].value_counts(dropna=False))
display(dim_trade_routes.head(20))
print(DIM_OUTPUT_PATH)


route_mode
direct               1024
forced_chokepoint     997
unmapped              460
Name: count, dtype: int64

route_group
OTHER_ROUTE               547
DIRECT_OR_UNCLASSIFIED    475
UNMAPPED                  460
GIBRALTAR_EXPOSED         338
SUEZ_EXPOSED              334
BLACK_SEA_EXIT_EXPOSED    145
PANAMA_EXPOSED             78
HORMUZ_EXPOSED             64
MALACCA_EXPOSED            37
ATLANTIC_ROUTE              2
CAPE_DIVERSION_RISK         1
Name: count, dtype: int64

,reporter_iso3,partner_iso3,partner2_iso3,reporter_port,partner2_port,partner_port,reporter_basin,partner_basin,distance_km,sea_distance_km,sea_distance_direct_km,sea_distance_forced_km,main_chokepoint,route_group,route_mode,route_basis,route_confidence,route_applicability_status,mot_codes_seen,route_path_coords
0,BGR,AGO,None,Burgas,None,Cabinda,NaN,NaN,5558.37,10422.16,10422.16,10422.16,Gibraltar Strait,GIBRALTAR_EXPOSED,forced_chokepoint,NaN,NaN,NaN,NaN,NaN
1,BGR,ALB,None,Burgas,None,Shengjin,NaN,NaN,653.87,1572.86,1572.86,NaN,Other / Unclassified,OTHER_ROUTE,direct,NaN,NaN,NaN,NaN,NaN
2,BGR,ARE,None,Burgas,None,Abu Zaby,NaN,NaN,3169.30,7275.16,7275.16,7275.16,Suez Canal,SUEZ_EXPOSED,forced_chokepoint,NaN,NaN,NaN,NaN,NaN
3,BGR,ARG,None,Burgas,None,La Plata,NaN,NaN,12196.58,13456.78,13456.78,NaN,Other / Unclassified,OTHER_ROUTE,direct,NaN,NaN,NaN,NaN,NaN
4,BGR,ARM,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Missing port mapping,UNMAPPED,unmapped,NaN,NaN,NaN,NaN,NaN
5,BGR,ATG,None,Burgas,None,St Johns,NaN,NaN,8679.31,9524.19,9524.19,NaN,Other / Unclassified,OTHER_ROUTE,direct,NaN,NaN,NaN,NaN,NaN
6,BGR,AUS,None,Varna,None,Fremantle,NaN,NaN,12212.65,15425.87,13871.09,15425.87,Suez Canal,SUEZ_EXPOSED,forced_chokepoint,NaN,NaN,NaN,NaN,NaN
7,BGR,AUT,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Missing port mapping,UNMAPPED,unmapped,NaN,NaN,NaN,NaN,NaN
8,BGR,AZE,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Missing port mapping,UNMAPPED,unmapped,NaN,NaN,NaN,NaN,NaN
9,BGR,BEL,None,Varna,None,Antwerpen,NaN,NaN,1980.03,6136.35,6136.35,NaN,Other / Unclassified,OTHER_ROUTE,direct,NaN,NaN,NaN,NaN,NaN


/Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/comtrade/dim_trade_routes.parquet


In [ ]:
# qa checks for neighbouring-country sanity and basin/chokepoint consistency
qa_pairs = [
    ("ROU", "BGR"),
    ("ROU", "UKR"),
    ("ROU", "GRC"),
    ("ROU", "ITA"),
    ("BGR", "TUR"),
    ("BGR", "ROU"),
    ("BGR", "GRC"),
    ("BGR", "ITA"),
]

qa_df = (
    dim_trade_routes[
        dim_trade_routes[["reporter_iso3", "partner_iso3"]].apply(tuple, axis=1).isin(qa_pairs)
    ]
    .sort_values(["reporter_iso3", "partner_iso3"])
    .copy()
)

display(
    qa_df[
        [
            "reporter_iso3",
            "partner_iso3",
            "reporter_port",
            "partner_port",
            "reporter_basin",
            "partner_basin",
            "main_chokepoint",
            "route_group",
            "route_mode",
            "distance_km",
            "sea_distance_direct_km",
            "sea_distance_forced_km",
        ]
    ]
)

suspicious_neighbour_routes = dim_trade_routes[
    (dim_trade_routes["distance_km"] <= 2500)
    & (dim_trade_routes["main_chokepoint"].isin(["Suez Canal", "Hormuz Strait", "Bab el-Mandeb", "Malacca Strait", "Panama Canal"]))
].copy()

display(
    suspicious_neighbour_routes[
        [
            "reporter_iso3",
            "partner_iso3",
            "reporter_basin",
            "partner_basin",
            "main_chokepoint",
            "route_mode",
            "distance_km",
        ]
    ].sort_values(["distance_km", "reporter_iso3", "partner_iso3"]).head(50)
)


In [11]:
# visualize a sample of computed routes on an interactive map
sample_size = 30
plot_df = dim_trade_routes[
    dim_trade_routes["route_path_coords"].map(lambda x: isinstance(x, list) and len(x) > 1)
].head(sample_size).copy()

if plot_df.empty:
    print("No route geometries available to plot.")
else:
    route_map = folium.Map(location=[20, 0], zoom_start=2, tiles="CartoDB positron")

    for row in plot_df.itertuples(index=False):
        coords_latlon = [[pt[1], pt[0]] for pt in row.route_path_coords]
        color = "#d62728" if row.route_mode == "forced_chokepoint" else "#1f77b4"
        popup = (
            f"{row.reporter_iso3} ({row.reporter_port}) -> {row.partner_iso3} ({row.partner_port})<br>"
            f"mode={row.route_mode}, cp={row.main_chokepoint}<br>"
            f"sea_km={row.sea_distance_km}"
        )
        folium.PolyLine(coords_latlon, color=color, weight=2.5, opacity=0.85, popup=popup).add_to(route_map)

        if coords_latlon:
            folium.CircleMarker(coords_latlon[0], radius=3, color="#2ca02c", fill=True).add_to(route_map)
            folium.CircleMarker(coords_latlon[-1], radius=3, color="#ff7f0e", fill=True).add_to(route_map)

    display(route_map)
    display(
        plot_df[
            [
                "reporter_iso3",
                "partner_iso3",
                "reporter_basin",
                "partner_basin",
                "route_mode",
                "main_chokepoint",
                "sea_distance_direct_km",
                "sea_distance_forced_km",
                "sea_distance_km",
            ]
        ].head(10)
    )

,reporter_iso3,partner_iso3,reporter_basin,partner_basin,route_mode,main_chokepoint,sea_distance_direct_km,sea_distance_forced_km,sea_distance_km
1488,ESP,AGO,ATLANTIC,OTHER,direct,NaN,5648.64,NaN,5648.64
1489,ESP,AGO,ATLANTIC,OTHER,direct,NaN,5648.64,NaN,5648.64
1490,ESP,ALB,MEDITERRANEAN,MEDITERRANEAN,direct,NaN,1810.40,NaN,1810.40
1491,ESP,ALB,MEDITERRANEAN,MEDITERRANEAN,direct,NaN,1810.40,NaN,1810.40
1492,ESP,ARE,MEDITERRANEAN,GULF,forced_chokepoint,Suez Canal,8422.40,8422.4,8422.40
1493,ESP,ARE,MEDITERRANEAN,GULF,forced_chokepoint,Suez Canal,8422.40,8422.4,8422.40
1494,ESP,ARE,MEDITERRANEAN,GULF,forced_chokepoint,Suez Canal,8422.40,8422.4,8422.40
1495,ESP,ARE,MEDITERRANEAN,GULF,forced_chokepoint,Suez Canal,8422.40,8422.4,8422.40
1496,ESP,ARE,MEDITERRANEAN,GULF,forced_chokepoint,Suez Canal,8422.40,8422.4,8422.40
1497,ESP,ARE,MEDITERRANEAN,GULF,forced_chokepoint,Suez Canal,8422.40,8422.4,8422.40
